# Graph RAG Introduction with topologic_fast

This notebook demonstrates the fundamentals of Graph-based Retrieval-Augmented Generation (GraphRAG) using topologic_fast. GraphRAG combines the power of graph data structures with modern AI to enable intelligent querying of spatial and topological data.

## What is GraphRAG?

GraphRAG is an approach that:
1. **Stores knowledge in graph structures** - vertices represent entities, edges represent relationships
2. **Embeds graph information** - converts graph data into vector representations
3. **Retrieves relevant context** - finds the most similar/relevant parts of the graph
4. **Generates responses** - uses retrieved context to answer questions

## Why use topologic_fast for GraphRAG?

- **High-performance graph operations** - fast creation and traversal of graphs
- **3D spatial awareness** - vertices have coordinates, enabling geometric queries
- **Architectural/engineering focus** - ideal for building models, spatial analysis
- **Clean API** - simple methods for graph manipulation

## Prerequisites

```bash
pip install chromadb sentence-transformers plotly
```

## 1. Setup and Imports

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go

print(f"topologic_fast imported successfully")

## 2. Creating Graphs in topologic_fast

In topologic_fast, graphs are created from vertices and edges. Each vertex has 3D coordinates, making them spatially aware.

In [ ]:
# Create vertices with 3D coordinates
v0 = tf.Vertex.ByCoordinates(0, 0, 0)
v1 = tf.Vertex.ByCoordinates(1, 0, 0)
v2 = tf.Vertex.ByCoordinates(1, 1, 0)
v3 = tf.Vertex.ByCoordinates(0, 1, 0)
v4 = tf.Vertex.ByCoordinates(0.5, 0.5, 1)

# Create edges connecting vertices
e0 = tf.Edge.ByStartVertexEndVertex(v0, v1)
e1 = tf.Edge.ByStartVertexEndVertex(v1, v2)
e2 = tf.Edge.ByStartVertexEndVertex(v2, v3)
e3 = tf.Edge.ByStartVertexEndVertex(v3, v0)
e4 = tf.Edge.ByStartVertexEndVertex(v0, v4)
e5 = tf.Edge.ByStartVertexEndVertex(v1, v4)
e6 = tf.Edge.ByStartVertexEndVertex(v2, v4)
e7 = tf.Edge.ByStartVertexEndVertex(v3, v4)

# Create graph from vertices and edges
vertices = [v0, v1, v2, v3, v4]
edges = [e0, e1, e2, e3, e4, e5, e6, e7]
graph = tf.Graph.ByVerticesEdges(vertices, edges)

print(f"Graph created:")
print(f"  Order (vertices): {graph.Order()}")
print(f"  Size (edges): {graph.Size()}")
print(f"  Density: {graph.Density():.3f}")

## 3. Creating Graphs from 3D Models

topologic_fast can automatically create dual graphs from topological structures like CellComplexes. This is powerful for building analysis - each room becomes a vertex, and shared walls become edges.

In [ ]:
# Create a simple 2x2 grid of rooms (cells)
rooms = []
room_data = [
    {"name": "Room A", "x": 0, "y": 0},
    {"name": "Room B", "x": 2, "y": 0},
    {"name": "Room C", "x": 0, "y": 2},
    {"name": "Room D", "x": 2, "y": 2},
]

for rd in room_data:
    room = tf.Cell.Box(rd["x"], rd["y"], 0, 2, 2, 3)  # 2x2x3 meter rooms
    rooms.append(room)

# Combine into a CellComplex
building = tf.CellComplex.ByCells(rooms)

print(f"Building CellComplex:")
print(f"  Cells (rooms): {building.NumCells()}")
print(f"  Total volume: {building.Volume():.1f} m³")

# Create dual graph - rooms become vertices, shared walls become edges
building_graph = tf.Graph.ByTopology(building)

print(f"\nBuilding connectivity graph:")
print(f"  Vertices (rooms): {building_graph.Order()}")
print(f"  Edges (connections): {building_graph.Size()}")
print(f"  Diameter: {building_graph.Diameter()} (max distance)")

## 4. Converting Graphs to Text for RAG

For GraphRAG, we need to convert graph structure into text that can be embedded. This function extracts vertex and edge information as natural language descriptions.

In [ ]:
def graph_to_texts(graph, vertex_labels=None, mantissa=2):
    """
    Convert a topologic_fast graph to text descriptions for embedding.
    
    Parameters:
    - graph: tf.Graph object
    - vertex_labels: Optional dict mapping vertex index to label
    - mantissa: decimal places for coordinates
    
    Returns:
    - List of text descriptions
    """
    texts = []
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    # Create vertex texts
    for i, vertex in enumerate(vertices):
        coords = vertex.Coordinates()
        label = vertex_labels.get(i, f"vertex_{i}") if vertex_labels else f"vertex_{i}"
        coord_str = f"({coords[0]:.{mantissa}f}, {coords[1]:.{mantissa}f}, {coords[2]:.{mantissa}f})"
        
        # Get adjacent vertices
        adjacent = graph.AdjacentVertices(vertex)
        adj_count = len(adjacent)
        
        text = f"Vertex '{label}' at coordinates {coord_str} has {adj_count} connection(s)"
        texts.append(text)
    
    # Create edge texts
    for i, edge in enumerate(edges):
        edge_verts = edge.Vertices()
        if len(edge_verts) >= 2:
            start_coords = edge_verts[0].Coordinates()
            end_coords = edge_verts[1].Coordinates()
            length = edge_verts[0].Distance(edge_verts[1])
            
            text = f"Edge {i} connects vertex at ({start_coords[0]:.{mantissa}f}, {start_coords[1]:.{mantissa}f}) " \
                   f"to vertex at ({end_coords[0]:.{mantissa}f}, {end_coords[1]:.{mantissa}f}), length: {length:.{mantissa}f}"
            texts.append(text)
    
    return texts

# Test with our simple graph
vertex_labels = {0: "origin", 1: "x-axis", 2: "xy-corner", 3: "y-axis", 4: "apex"}
texts = graph_to_texts(graph, vertex_labels)

print("Graph converted to text descriptions:")
print("=" * 50)
for text in texts:
    print(f"  {text}")

## 5. Graph Analysis for Context Retrieval

Understanding graph structure helps in retrieving relevant context. topologic_fast provides many analysis methods.

In [ ]:
def analyze_graph(graph):
    """Comprehensive graph analysis."""
    vertices = graph.Vertices()
    
    print("Graph Analysis")
    print("=" * 50)
    print(f"Order (# vertices): {graph.Order()}")
    print(f"Size (# edges): {graph.Size()}")
    print(f"Density: {graph.Density():.3f}")
    print(f"Diameter: {graph.Diameter()}")
    print(f"Max degree: {graph.MaximumDelta()}")
    print(f"Min degree: {graph.MinimumDelta()}")
    print(f"Is complete: {graph.IsComplete()}")
    print(f"Is bipartite: {graph.IsBipartite()}")
    
    print(f"\nDegree sequence: {graph.DegreeSequence()}")
    
    # Vertex degrees
    print(f"\nVertex degrees:")
    for i, v in enumerate(vertices):
        degree = graph.VertexDegree(v)
        coords = v.Coordinates()
        print(f"  Vertex {i} at ({coords[0]:.1f}, {coords[1]:.1f}, {coords[2]:.1f}): degree {degree}")

# Analyze our graph
analyze_graph(graph)

## 6. Pathfinding for Spatial Queries

Finding paths between vertices is essential for spatial queries in GraphRAG.

In [ ]:
# Find shortest path between two vertices
vertices = graph.Vertices()
v_start = vertices[0]  # origin
v_end = vertices[2]    # xy-corner

# Get distance (number of edges)
distance = graph.Distance(v_start, v_end)
print(f"Graph distance from origin to xy-corner: {distance} edge(s)")

# Get the actual path
path = graph.Path(v_start, v_end)
if path:
    path_verts = path.Vertices()
    print(f"\nPath (via {len(path_verts)} vertices):")
    for i, v in enumerate(path_verts):
        coords = v.Coordinates()
        label = vertex_labels.get(i, f"v{i}")
        print(f"  Step {i}: ({coords[0]:.1f}, {coords[1]:.1f}, {coords[2]:.1f})")

# Depth map from origin
depth_map = graph.DepthMap(v_start)
print(f"\nDepth map from origin: {depth_map}")

## 7. Visualizing the Graph with Plotly

In [ ]:
def visualize_graph_3d(graph, vertex_labels=None, title="3D Graph Visualization"):
    """Visualize a topologic_fast graph in 3D using Plotly."""
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    fig = go.Figure()
    
    # Draw edges
    for edge in edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) >= 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='blue', width=4),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw vertices
    coords = [v.Coordinates() for v in vertices]
    x = [c[0] for c in coords]
    y = [c[1] for c in coords]
    z = [c[2] for c in coords]
    
    # Color by degree
    degrees = [graph.VertexDegree(v) for v in vertices]
    
    # Labels
    labels = [vertex_labels.get(i, f"v{i}") for i in range(len(vertices))] if vertex_labels else [f"v{i}" for i in range(len(vertices))]
    hover_text = [f"{labels[i]}<br>Degree: {degrees[i]}" for i in range(len(vertices))]
    
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers+text',
        marker=dict(
            size=15,
            color=degrees,
            colorscale='Viridis',
            colorbar=dict(title='Degree'),
            line=dict(color='black', width=1)
        ),
        text=labels,
        textposition='top center',
        hovertext=hover_text,
        hoverinfo='text',
        name='Vertices'
    ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data'
        ),
        width=800,
        height=600
    )
    
    return fig

# Visualize our graph
fig = visualize_graph_3d(graph, vertex_labels, "Pyramid Graph - Color by Degree")
fig.show()

## 8. Visualize Building Graph

In [ ]:
# Visualize the building and its connectivity graph
room_labels = {i: room_data[i]["name"] for i in range(len(room_data))}

# Create figure with both geometry and graph
fig = go.Figure()

# Draw room boundaries (top-down view as filled rectangles)
colors = ['#87CEEB', '#90EE90', '#FFD700', '#FFB6C1']

for i, rd in enumerate(room_data):
    x0, y0 = rd["x"], rd["y"]
    x = [x0, x0+2, x0+2, x0, x0]
    y = [y0, y0, y0+2, y0+2, y0]
    
    fig.add_trace(go.Scatter(
        x=x, y=y,
        fill='toself',
        fillcolor=colors[i],
        line=dict(color='black', width=2),
        name=rd["name"],
        hoverinfo='name'
    ))

# Draw graph edges
graph_edges = building_graph.Edges()
for edge in graph_edges:
    edge_verts = edge.Vertices()
    if len(edge_verts) >= 2:
        p1 = edge_verts[0].Coordinates()
        p2 = edge_verts[1].Coordinates()
        fig.add_trace(go.Scatter(
            x=[p1[0], p2[0]],
            y=[p1[1], p2[1]],
            mode='lines',
            line=dict(color='red', width=4),
            showlegend=False,
            hoverinfo='skip'
        ))

# Draw graph vertices
graph_verts = building_graph.Vertices()
coords = [v.Coordinates() for v in graph_verts]
x = [c[0] for c in coords]
y = [c[1] for c in coords]

fig.add_trace(go.Scatter(
    x=x, y=y,
    mode='markers',
    marker=dict(size=20, color='red', line=dict(color='darkred', width=2)),
    name='Room Centers',
    hovertext=[rd["name"] for rd in room_data],
    hoverinfo='text'
))

fig.update_layout(
    title='Building Floor Plan with Connectivity Graph',
    xaxis=dict(title='X (m)', scaleanchor='y', scaleratio=1),
    yaxis=dict(title='Y (m)'),
    width=700,
    height=600,
    showlegend=True
)

fig.show()

## Summary

In this notebook, we covered the fundamentals of GraphRAG with topologic_fast:

1. **Creating graphs** from vertices and edges with `tf.Graph.ByVerticesEdges()`
2. **Generating graphs from 3D models** using `tf.Graph.ByTopology()`
3. **Converting graphs to text** for embedding in RAG systems
4. **Analyzing graph structure** with methods like `Density()`, `Diameter()`, etc.
5. **Pathfinding** with `Distance()` and `Path()`
6. **Visualization** using Plotly

### Next Steps

In the next notebook (GraphRAG02), we will:
- Set up ChromaDB for vector storage
- Embed graph information using sentence transformers
- Implement similarity search for graph retrieval
- Build a complete GraphRAG pipeline